# Demo - Translation with Positive and Negative Constraints

English-to-Russian translation where phrases can be applied as positive or negative constraints — forcing them to appear in the translation, or blocking them from appearing at all. Decoding uses my variation and interpretation of the original Grid Beam Search (Hokamp & Liu, 2017).

Three steps:

1. Edit the **Input** cell: write your own sentence and constraints, or copy-paste one of the ready-made examples from the *Example inputs* section below.
2. Make sure you are on a GPU runtime for best performance(preferably a Colab T4), then run the setup cell(Cell 1) and all cells below it. Setup takes about a minute.
3. For further translations in the same session, just edit the Input cell(Cell 2) and re-run it along with the cells below it. Each translation takes about 1-2 seconds.

**Reading the output:** the Translation Output cell prints a check for each constraint.

- Required phrases: `Present` — the phrase appears in the constrained translation; `??` — the phrase is missing, i.e. the constraint was not satisfied.
- Blocked phrases: `OK` — the phrase was successfully kept out; `FOUND` — the phrase still appears in the translation.


## Example inputs - pick one and paste it into the Input cell below

`SENTENCE` is the source English sentence.

`CONSTRAINTS` are Russian phrases that must appear in the target translation.

`NEGATIVE_CONSTRAINTS` are Russian phrases that must not appear as exact token sequences.

The examples are organized by the constraint use case they demonstrate. Constraints do not have to be nouns: any contiguous phrase works, and several constraints can be stacked. Placement stays the model's job, while the decoder enforces required phrases and filters blocked phrases during search.

### Section 1: Domain-Specific Jargon (Legal, Medical, and Technical Terminology)

Here, constraints preserve domain-specific jargon that a translation model might otherwise replace with simpler layperson wording to improve fluency.

**Example 1**

```python
SENTENCE = "During the commercial litigation hearing, the attorney cited a binding precedent from the appellate court to argue that the supplier could not enforce the disputed contract after repeatedly failing to deliver the goods on time."
CONSTRAINTS = [
    "адвокат",  # attorney; Avoiding: юрист (lawyer), защитник (counsel)
    "прецедент",  # precedent; Avoiding: предыдущее решение суда (previous court decision)
]
NEGATIVE_CONSTRAINTS = []
```

**Example 2**

```python
SENTENCE = "Before trial, the company director filed a motion to dismiss the shareholder lawsuit, arguing that the complaint failed to allege facts showing a breach of fiduciary duty."
CONSTRAINTS = [
    "иск акционеров",  # shareholder lawsuit; Avoiding: иск владельцев компании (lawsuit by company owners)
    "нарушение фидуциарной обязанности",  # breach of fiduciary duty; Avoiding: нарушение доверительных обязательств (breach of trust obligations), неисполнение обязанности действовать в чужих интересах (failure to act in another's interests)
]
NEGATIVE_CONSTRAINTS = []
```

**Example 3**

```python
SENTENCE = "The patient presented to the emergency department with bilateral lower extremity edema and dyspnea, prompting the physician to admit her for further evaluation."
CONSTRAINTS = [
    "двусторонний отек нижних конечностей",  # bilateral lower extremity edema; Avoiding: отек обеих ног (swelling of both legs), опухшие ноги (swollen legs)
    "одышка",  # dyspnea; Avoiding: затрудненное дыхание (difficulty breathing), нехватка воздуха (feeling short of air)
]
NEGATIVE_CONSTRAINTS = []
```

**Example 4**

```python
SENTENCE = "The patient's electrocardiogram findings were consistent with an acute myocardial infarction, requiring immediate intervention by the cardiology team."
CONSTRAINTS = [
    "электрокардиограмма",  # electrocardiogram; Avoiding: ЭКГ (the abbreviation, "ECG"), запись работы сердца (heart activity tracing)
    "инфаркт миокарда",  # myocardial infarction; Avoiding: сердечный приступ (heart attack)
]
NEGATIVE_CONSTRAINTS = []
```

**Example 5**

```python
SENTENCE = "The software engineer traced the intermittent failures to a race condition in the multithreaded scheduler, which caused nondeterministic behavior under heavy load."
CONSTRAINTS = [
    "состояние гонки",  # race condition; Avoiding: условие гонки (word-for-word calque of "race condition"), ошибка синхронизации (synchronization error)
    "недетерминированное поведение",  # nondeterministic behavior; Avoiding: непредсказуемое поведение (unpredictable behavior)
]
NEGATIVE_CONSTRAINTS = []
```

**Example 6**

```python
SENTENCE = "The database administrator optimized the query by adding a composite index, significantly reducing the execution time for large join operations."
CONSTRAINTS = [
    "составной индекс",  # composite index; Avoiding: композитный индекс (anglicized "composite index"), индекс по нескольким столбцам (index on several columns)
    "операция соединения"  # join operation; Avoiding: операция объединения (union operation, a different SQL concept), объединение таблиц (combining tables)
]
NEGATIVE_CONSTRAINTS = []
```

### Section 2: Named Entities (People, Places, and Organizations)

These constraints fix names of people, places, and organizations to one standard Russian form instead of an alternate spelling or transliteration.

**Example 1**

```python
SENTENCE = "Jacinda Ardern delivered a speech to some officials from Wellington."
CONSTRAINTS = [
    "Джасинда Ардерн",  # Jacinda Ardern; Avoiding: Ясинда Ардерн ("Yasinda", wrong first letter), Джасинда Ардёрн ("Ardyorn", wrong vowel)
    "Веллингтон"  # Wellington; Avoiding: Уэллингтон ("Wellington", variant spelling), Велингтон ("Wellington", misspelled with one "л"(l))
]
NEGATIVE_CONSTRAINTS = []
```

**Example 2**

```python
SENTENCE = "The United Nations hosted a climate policy meeting in Geneva."
CONSTRAINTS = [
    "Организация Объединенных Наций",  # United Nations; Avoiding: ООН (the abbreviation, "UN"), Объединенные Нации ("United Nations", clipped form)
    "Женева"  # Geneva; Avoiding: Дженева ("Geneva", English-style spelling), Генева ("Geneva", letter-by-letter spelling)
]
NEGATIVE_CONSTRAINTS = []
```

**Example 3**

```python
SENTENCE = "There was a diplomatic reception at the Élysée Palace in Paris."
CONSTRAINTS = [
    "Елисейский дворец",  # Élysée Palace; Avoiding: дворец Элизе ("Élysée Palace", half-transliterated), Элизейский дворец ("Élysée Palace", misspelled adjective)
    "Париж",  # Paris; Avoiding; Парис ("Paris", English-style spelling), Пари ("Paris", French-pronunciation spelling)
]
NEGATIVE_CONSTRAINTS = []
```

### Section 3: Brand and Product Name Consistency

These constraints keep brand and product names in their official Latin-script form instead of a translation or Cyrillic transliteration.

**Example 1**

```python
SENTENCE = "Google added new coding features to Gemini Advanced and integrated them with Google Workspace."
CONSTRAINTS = [
    "Google",  # Google; Avoiding; Гугл (Cyrillic transliteration)
    "Gemini Advanced",  # Gemini Advanced; Avoiding: Близнецы (literal translation, "the Twins"), Джемини Адвансед (Cyrillic transliteration)
]
NEGATIVE_CONSTRAINTS = []
```


**Example 2**

```python
SENTENCE = "Microsoft released new enterprise features for Copilot and integrated them with Microsoft 365."
CONSTRAINTS = [
    "Microsoft",  # Microsoft; Avoiding: Майкрософт (Cyrillic transliteration)
    "Copilot",  # Copilot; Avoiding: второй пилот (literal translation, "co-pilot"), Копайлот (Cyrillic transliteration)
]
NEGATIVE_CONSTRAINTS = []
```

**Example 3**

```python
SENTENCE = "Apple introduced new spatial computing features for Vision Pro and expanded support across its developer tools."
CONSTRAINTS = [
    "Apple",  # Apple; Avoiding: Эппл (Cyrillic transliteration), Яблоко (literal translation, the fruit "apple")
    "Vision Pro",  # Vision Pro; Avoiding: Вижн Про (Cyrillic transliteration), Видение Про (literal translation of "Vision")
]
NEGATIVE_CONSTRAINTS = []
```

### Section 4: Glossary Compliance

These constraints force the approved glossary term when the source word has several plausible Russian translations.

**Example 1**

```python
SENTENCE = "Each customer account is provisioned as a separate tenant with its own users, permissions, and workspace settings."
CONSTRAINTS = [
    "тенант",  # tenant; Avoiding: арендатор (property renter), жилец (occupant of a home)
]
NEGATIVE_CONSTRAINTS = []
```

**Example 2**

```python
SENTENCE = "The API gateway routes each request to the correct endpoint based on the path and method."
CONSTRAINTS = [
    "эндпоинт",  # endpoint; Avoiding: конечная точка (literal "end point")
]
NEGATIVE_CONSTRAINTS = []
```

**Example 3**

```python
SENTENCE = "The merchant received a chargeback after the customer disputed the card transaction."
CONSTRAINTS = [
    "чарджбэк",  # chargeback; Avoiding: возврат платежа (payment refund), возврат средств (refund of funds)
]
NEGATIVE_CONSTRAINTS = []
```

**Example 4**

```python
SENTENCE = "The support team reviewed the customer's open ticket before escalating the issue to engineering."
CONSTRAINTS = [
    "заявка",  # ticket: Avoiding; билет (travel or event ticket), тикет (transliterated "ticket")
]
NEGATIVE_CONSTRAINTS = []
```

### Section 5: Content Sanitization: Blocked Phrases and Tone Control

These constraints prevent unwanted target-language words or phrases from appearing in the translation, especially when the source contains wording or tone that should not be carried over to target langauge directly.

It could be harsh wording, competitor names or sensitive information that should be softened or moderated.

**Example 1**

```python
SENTENCE = "The customer called the delayed rollout a stupid mess and said the support team was useless."
CONSTRAINTS = []
NEGATIVE_CONSTRAINTS = [
    "тупой",  # stupid / dumb; Instead may use: "неудачный" (unsuccessful) or "плохо спланированный" (poorly planned).
    "бесполезный",  # useless; Instead may use: "неэффективный" (ineffective) or "недостаточно полезный" (not useful enough).
]
```

**Example 2**

```python
SENTENCE = "Several customers said they compared our collaboration platform with Slack before choosing our product."
CONSTRAINTS = []
NEGATIVE_CONSTRAINTS = [
    "Slack",  # competitor name; Instead may use: "конкурирующая платформа" (competing platform) or "альтернативный сервис" (alternative service).
]
```

**Example 3**

This final example demonstrates how negative constraints could be used in tandem with positive constraints through an example scenario. 


```python
SENTENCE = "The registration form included the applicant's passport number AB1234567 and date of birth May 12, 1998."
CONSTRAINTS = [
    "[СКРЫТО]", # [REDACTED]; Avoiding; Passport numbers
]
NEGATIVE_CONSTRAINTS = [
    "AB1234567",  # Detected passport number; Instead may use: "[СКРЫТО]" ([REDACTED]) because we promoted it as a constraint or "скрытые паспортные данные" (hidden passport details) because it is probable.
    "May 12, 1998",  # Detected date of birth; Instead may use: "[СКРЫТО]" ([REDACTED]) or "скрытая дата рождения" (hidden date of birth) because it is probable.
]
```

In [ ]:
## Cell 1: Setup -- run once (Setup complete -- model ready on cuda.)
%pip install -q transformers torch sentencepiece "numpy>=2" "accelerate>=0.26.0"

from transformers import MarianMTModel, MarianTokenizer
from transformers.modeling_outputs import BaseModelOutput
import torch
import torch.nn.functional as F

NUM_BEAMS = 4
MAX_LENGTH = 128

tokenizer = MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-ru")
model = MarianMTModel.from_pretrained("Helsinki-NLP/opus-mt-en-ru")
special_ids = set(tokenizer.all_special_ids)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
model.to(device)
model.eval()


def encode_target(text, **kwargs):
    """Tokenize Russian text with the *target* SentencePiece model.

    Constraints are forced into the output, so they must be segmented
    exactly the way the decoder emits tokens -- see Section 3 of
    `Final Research Draft.ipynb` for why this matters.
    """
    try:
        return tokenizer(text_target=text, **kwargs)
    except TypeError:  # transformers < 4.22
        with tokenizer.as_target_tokenizer():
            return tokenizer(text, **kwargs)


def grid_beam_search(
    input_ids,
    attention_mask,
    constraints,
    negative_constraints=(),
    num_beams=NUM_BEAMS,
    max_length=MAX_LENGTH,
    length_penalty=1.0,
):
    eos_id = tokenizer.eos_token_id
    start_id = model.config.decoder_start_token_id
    pad_id = model.config.pad_token_id

    ctoks = []
    for c in constraints:
        toks = [
            int(t)
            for t in c["token_ids"].tolist()
            if int(t) not in special_ids
        ]
        if toks:
            ctoks.append(toks)

    btoks = []
    for c in negative_constraints:
        toks = [
            int(t)
            for t in c["token_ids"].tolist()
            if int(t) not in special_ids
        ]
        if toks:
            btoks.append(toks)

    num_constraints = len(ctoks)
    total_constraint_tokens = sum(len(t) for t in ctoks)

    src_len = int(attention_mask.sum().item())
    max_steps = min(max_length, 2 * src_len + total_constraint_tokens + 16)

    # the encoder runs exactly once; its states are re-used every step
    with torch.no_grad():
        enc = model.get_encoder()(input_ids=input_ids, attention_mask=attention_mask)

    def coverage(h):
        cov = sum(len(ctoks[j]) for j in h["done"])
        if h["open"] is not None:
            cov += h["open"][1]
        return cov

    def violates_blocklist(ids, tok):
        candidate = ids + [tok]
        for banned in btoks:
            if len(candidate) >= len(banned) and candidate[-len(banned):] == banned:
                return True
        return False

    def blocked_next_tokens(ids):
        blocked = set()
        for banned in btoks:
            prefix = banned[:-1]
            if not prefix or (len(ids) >= len(prefix) and ids[-len(prefix):] == prefix):
                blocked.add(banned[-1])
        return blocked

    init = {
        "ids": [start_id],
        "score": 0.0,
        "done": frozenset(),
        "open": None,
    }

    grid = {0: [init]}
    completed = []

    for step in range(max_steps):
        remaining = max_steps - step

        # prune hypotheses that can no longer cover all constraints in time
        active = [
            h
            for hs in grid.values()
            for h in hs
            if (total_constraint_tokens - coverage(h)) <= remaining
        ]

        if not active:
            break

        dec = torch.tensor(
            [h["ids"] for h in active],
            dtype=torch.long,
            device=device,
        )

        exp_enc = BaseModelOutput(
            last_hidden_state=enc.last_hidden_state.expand(len(active), -1, -1)
        )

        with torch.no_grad():
            dec_out = model.model(
                encoder_outputs=exp_enc,
                attention_mask=attention_mask.expand(len(active), -1),
                decoder_input_ids=dec,
            )

            last_hidden = dec_out.last_hidden_state[:, -1, :]
            logits = model.lm_head(last_hidden) + model.final_logits_bias

        logp = F.log_softmax(logits, dim=-1)
        new_grid = {}

        def add(h):
            new_grid.setdefault(coverage(h), []).append(h)

        for idx, h in enumerate(active):
            c = coverage(h)
            lp = logp[idx]
            must_cover = (total_constraint_tokens - c) >= remaining

            # CONTINUE: an open constraint must be finished before anything else
            if h["open"] is not None:
                j, p = h["open"]
                tok = ctoks[j][p]
                if violates_blocklist(h["ids"], tok):
                    continue

                nxt = p + 1
                fin = h["done"] | {j} if nxt == len(ctoks[j]) else h["done"]

                add(
                    {
                        "ids": h["ids"] + [tok],
                        "score": h["score"] + lp[tok].item(),
                        "done": fin,
                        "open": None if nxt == len(ctoks[j]) else (j, nxt),
                    }
                )
                continue

            # GENERATE: free tokens, only while there is still time to cover the rest
            if not must_cover:
                gen_lp = lp.clone()
                gen_lp[pad_id] = float("-inf")

                if c < total_constraint_tokens:
                    gen_lp[eos_id] = float("-inf")

                for tok in blocked_next_tokens(h["ids"]):
                    gen_lp[tok] = float("-inf")

                top_lp, top_ids = torch.topk(gen_lp, num_beams)

                for k in range(num_beams):
                    s = top_lp[k].item()

                    if s == float("-inf"):
                        continue

                    tok = int(top_ids[k])

                    nh = {
                        "ids": h["ids"] + [tok],
                        "score": h["score"] + s,
                        "done": h["done"],
                        "open": None,
                    }

                    if tok == eos_id:
                        completed.append(nh)
                    else:
                        add(nh)

            # START: open any not-yet-covered constraint
            for j in range(num_constraints):
                if j in h["done"]:
                    continue

                tok = ctoks[j][0]
                if violates_blocklist(h["ids"], tok):
                    continue

                nxt = 1
                fin = h["done"] | {j} if nxt == len(ctoks[j]) else h["done"]

                add(
                    {
                        "ids": h["ids"] + [tok],
                        "score": h["score"] + lp[tok].item(),
                        "done": fin,
                        "open": None if nxt == len(ctoks[j]) else (j, nxt),
                    }
                )

        grid = {}

        for c, hs in new_grid.items():
            hs.sort(key=lambda x: x["score"], reverse=True)
            # keep the top-k *distinct* hypotheses per coverage level
            seen, uniq = set(), []
            for h in hs:
                key = (tuple(h["ids"]), h["open"], h["done"])
                if key in seen:
                    continue
                seen.add(key)
                uniq.append(h)
                if len(uniq) == num_beams:
                    break
            grid[c] = uniq

    def norm(h):
        length = len(h["ids"]) - 1
        return h["score"] / (length ** length_penalty) if length > 0 else h["score"]

    if completed:
        pool = completed
    elif total_constraint_tokens in grid and grid[total_constraint_tokens]:
        pool = grid[total_constraint_tokens]
    else:
        pool = grid[max(grid)]

    best = max(pool, key=norm)
    return torch.tensor([best["ids"]], dtype=torch.long, device=device)


def encode_constraints(phrases):
    cons = []
    for phrase in phrases:
        ids = encode_target(
            phrase, add_special_tokens=False, return_tensors="pt"
        )["input_ids"][0].to(device)
        cons.append({"token_ids": ids})
    return cons


def translate(sentence, constraints=(), negative_constraints=()):
    """Translate one English sentence while enforcing target-side phrases.

    `constraints` must appear in the output; `negative_constraints` must not
    appear as exact target-token sequences.
    """
    enc = tokenizer(
        [sentence], return_tensors="pt", truncation=True, max_length=MAX_LENGTH
    ).to(device)
    cons = encode_constraints(constraints)
    blocked = encode_constraints(negative_constraints)
    out_ids = grid_beam_search(
        enc["input_ids"],
        enc["attention_mask"],
        cons,
        negative_constraints=blocked,
    )
    return tokenizer.batch_decode(out_ids, skip_special_tokens=True)[0]


print(f"Setup complete -- model ready on {device}.")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/803k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/1.08M [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.60M [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/307M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/307M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

Setup complete -- model ready on cuda.


In [ ]:
## Cell 2: Input
SENTENCE = "During the commercial litigation hearing, the attorney cited a binding precedent from the appellate court to argue that the supplier could not enforce the disputed contract after repeatedly failing to deliver the goods on time."

# Russian phrases that must appear in the translation.
CONSTRAINTS = [
    "адвокат",  # attorney; layman: lawyer
    "прецедент",  # precedent; layman: previous court decision
]

# Russian phrases that must not appear in the translation.
NEGATIVE_CONSTRAINTS = []


In [ ]:
## Cell 3: Translation Output
unconstrained = translate(SENTENCE)
constrained = translate(SENTENCE, CONSTRAINTS, NEGATIVE_CONSTRAINTS)

print(f"Source           : {SENTENCE}")
print(f"Plain translation: {unconstrained}")
print(f"With constraints : {constrained}")

if CONSTRAINTS:
    print("\nRequired phrases:")
    for phrase in CONSTRAINTS:
        mark = "Present " if phrase in constrained else "?? "
        print(f"  {mark}{phrase!r}")

if NEGATIVE_CONSTRAINTS:
    print("\nBlocked phrases:")
    for phrase in NEGATIVE_CONSTRAINTS:
        mark = "OK " if phrase not in constrained else "FOUND "
        print(f"  {mark}{phrase!r}")


Source           : Apple introduced new spatial computing features for Vision Pro and expanded support across its developer tools.
Plain translation: Apple ввела новые пространственные вычислительные функции для Vision Pro и расширила поддержку своих разработчиков.
With constraints : Apple ввела новые пространственные вычислительные функции для Vision Pro и расширила поддержку своих разработчиков.

Required phrases:
  OK 'Apple'
  OK 'Vision Pro'
